# 2 · Weight estimation

Class I sizing, the snowball, and class II on the 737-800's geometry.
Lesson: [02](../docs/lessons/02-weight-estimation.md) ·
Theory: [weight estimation](../docs/weight_estimation.md).

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from acdesign import plotting as P
from acdesign.requirements import B737_800_MISSION as M
from acdesign.units import FT, G0, LB, NMI

P.use_style()
np.set_printoptions(precision=4, suppress=True)
sys.path.insert(0, "../validation")  # cited reference data and the reference aircraft
from reference import b737_800

R = b737_800()
from acdesign.weight_estimation import (
    breguet_cruise_fraction, size_class_one, max_lift_to_drag,
    component_masses, group_masses, operating_empty_mass,
)
from reference_aircraft import class_two_inputs, crew_mass, wetted_areas

Breguet: the cruise fraction for the design range, at three L/D values.

In [2]:
V, tsfc = 231.3, 17.12e-6  # Mach 0.78 at FL350; AERO-03 cruise TSFC [kg/(N s)]
for LD in (15.0, 16.5, 18.0):
    f = breguet_cruise_fraction(M.design_range, V, LD, tsfc)
    print(f"L/D {LD}: W_end/W_start = {f:.4f}, cruise burns {1 - f:.1%}")

L/D 15.0: W_end/W_start = 0.7781, cruise burns 22.2%
L/D 16.5: W_end/W_start = 0.7960, cruise burns 20.4%
L/D 18.0: W_end/W_start = 0.8113, cruise burns 18.9%


Class I with the reference's wetted-area ratio; then the growth factor by adding payload.

In [3]:
from dataclasses import replace

LD = max_lift_to_drag(R["wing_aspect_ratio"], sum(wetted_areas().values()) / R["wing_area"])
r = size_class_one(M, V, LD, tsfc, 18.0e-6, crew=crew_mass())
r2 = size_class_one(replace(M, cargo=1000.0), V, LD, tsfc, 18.0e-6, crew=crew_mass())
print(f"(L/D)max {LD:.2f}; W0 = {r.W0:,.0f} kg ({r.W0 / R['mtow'] - 1:+.1%} vs MTOW), {len(r.history) - 1} iterations")
print(f"growth factor dW0/dWp = {(r2.W0 - r.W0) / 1000:.2f}")

(L/D)max 19.11; W0 = 80,395 kg (+1.7% vs MTOW), 11 iterations
growth factor dW0/dWp = 4.18


Class II on the published 737-800 geometry, no calibration:

In [4]:
m = component_masses(class_two_inputs())
for g, v in group_masses(m).items():
    print(f"{g:16s} {v:8,.0f} kg")
oew = operating_empty_mass(m)
print(f"OEW {oew:,.0f} kg vs {R['oew']:,.0f} kg published ({oew / R['oew'] - 1:+.1%})")

structure          18,900 kg
propulsion          5,092 kg
systems            10,644 kg
operating_items     1,935 kg
OEW 36,571 kg vs 41,413 kg published (-11.7%)
